In [ ]:
import torch
import torch.nn as nn

In [ ]:
inputs = torch.tensor([
    [0.34,0.34,0.23],        # how
    [0.06,0.27,0.12],       # are
    [0.11,0.42,0.35],   # you
    [0.61,0.27,0.84],   # ?
])

# Add batch dimension
inputs = inputs.unsqueeze(0)
print(f"Inputs shape: {inputs.shape}")

# Basic Self-Attention
softmax(QK_t/sqrt(d_k))V

In [ ]:
import math

print("\nAll Query Vector Case")
print("------------------------")

inputs2 = torch.cat((inputs, inputs))
print(f"\n2 batch input case: {inputs2.shape}")

Q,K,V = inputs2, inputs2, inputs2   # (2,4,3)  
K_t = K.transpose(-2, -1)           # (2,3,4)

# Dot product self-attention. If there are x tokens in a sentence, then we create a score matrix of (x,x). Each token is paired with each other token and itself to product x**2 dor products along the embedding dimension
S = (Q @ K_t) / math.sqrt(Q.shape[-1])  # (2,4,3) @ (2,3,4)

# Visualization: Matrix Mutliply of Q and K_t => element wise multiplication of a row of left matrix and a col of right matrix => (row from Q) · (column from K_t)

# Q=K=V: (4,3) => 
# [
# 'How' (x1,y1,z1)
# 'are' (x2,y2,z2)
# 'you' (x3,y3,z3)
# '?'   (x4,y4,z4)
# ]

# Each Q needs to attend against each key => K_t

# K_t: (3,4)
# [
# 'How',  'are',  'you',   "?"
#  x1      x2      x3      x4
#  y1      y2      y3      y4
#  z1      z2      z3      z4
# ]

# S=A (shape wise): (4,4)
#        'How',      'are',      'you',      "?"
# [
# 'How'  (how,how)  (how,are)  (how,you)  (how,?)
# 'are'  (are,how)  (are,are)  (are,you)  (are,?)  
# 'you'  (you,how)  (you,are)  (you,you)  (you,?)  
# '?'    (?,how)    (?,are)    (?,you)    (?,?)  
# ]


A = torch.softmax(S, dim=-1)
C = A @ V

#        'How',      'are',      'you',      "?"
# [
# 'How'  (how,how)  (how,are)  (how,you)  (how,?)
# 'are'  (are,how)  (are,are)  (are,you)  (are,?)  
# 'you'  (you,how)  (you,are)  (you,you)  (you,?)  
# '?'    (?,how)    (?,are)    (?,you)    (?,?)  
# ]

# <<<<<<MATRIX MULTIPLY>>>>>>

# [
# 'How' (x1,y1,z1)
# 'are' (x2,y2,z2)
# 'you' (x3,y3,z3)
# '?'   (x4,y4,z4)
# ]

# C: (4,3)
# What is this doing conceptually?
# Again, we do element wise multiplicaiton of one row of attention weights (A) with one col of V
## First row of A => Attention weights of the word how with itself and every other word => 4 values
## First col of V => First dimension of all the words => 4 values
## Element wise multiplication: Creates the first contextualized dimension of the word 'how' based on the attention of the other words. And this happens then for the second dimension and then the third dimension

# [
# 'How' (attended x1, attended y1, attended z1)
# 'are' (...)
# 'you' (...)
# '?'   (...)
# ]
print("C:")
C

# Trainable Self-Attention

In [ ]:
X = inputs2   # (2,4,3)  
X.shape
torch.rand(3,5).shape

(X @ torch.rand(3,5)).shape

In [ ]:
import math

print("\nAll Query Vector Case")
print("------------------------")

inputs2 = torch.cat((inputs, inputs))
print(f"\n2 batch input case: {inputs2.shape}")

X = inputs2   # (2,4,3)  
dim = inputs.shape[-1]
dk = 5
dv = 10

Wq = nn.Parameter(torch.rand(dim,dk))  # (3,5) => (embedding dim, dk)
Wk = nn.Parameter(torch.rand(dim,dk))  # (3,5) => (embedding dim, dk)
Wv = nn.Parameter(torch.rand(dim,dv))  # (3,10) => (embedding dim, dv)

Q = X @ Wq  # (2,4,5)
K = X @ Wk  # (2,4,5)
V = X @ Wv  # (2,4,10)

# What these learnt projections mean?
# Q => becomes 2,4,5 from 2,4,3. Each token is projected to another space.
# For Q, it means what information from the other tokens am I trying to get -> search signal generator
# K => What information am I projecting about myself (for relevance)?
# V => What information do I really contribute

# Rest of the implementation is the same 
K_t = K.transpose(-2, -1)           # (2,5,4)
S = (Q @ K_t) / math.sqrt(Q.shape[-1])  # (2,4,5) @ (2,5,4) => (2,4,4)
A = torch.softmax(S, dim=-1)  # (2,4,4)
C = A @ V  # (2,4,10)
print("C:")
print(C.shape)